# Module 4: Parallel Fork-Join: Deploy to Amazon Bedrock AgentCore

**Pattern 2: Parallel Fork-Join**: Researcher → [Analyzer A, B, C via GraphBuilder] → Synthesizer. The three analyzers run in parallel.

![Parallel Fork-Join Runtime architecture](./architecture.png)

**What this notebook covers:**
1. Install the AgentCore CLI
2. Configure and deploy
3. Review `main.py`
4. Find the Runtime ARN
5. Invoke the deployed agent with boto3
6. Observability in CloudWatch
7. Cleanup

---

## Step 1: Install the AgentCore CLI

In [ ]:
!uv pip install --system bedrock-agentcore-starter-toolkit bedrock-agentcore strands-agents boto3

In [ ]:
import boto3, os

# Verify the shared AgentCore execution roles before deploying.
# Workshop Studio: pre-created by the workshop CloudFormation stack.
# Self-paced (full IAM access): deploy.py creates them on first run.

iam = boto3.client("iam")

for role_name, env_var in [
    ("workshop-agentcore-m4-runtime-role",      "AGENTCORE_RUNTIME_ROLE_ARN"),
    ("workshop-agentcore-m4-orchestrator-role", "AGENTCORE_ORCHESTRATOR_ROLE_ARN"),
]:
    try:
        arn = iam.get_role(RoleName=role_name)["Role"]["Arn"]
        os.environ[env_var] = arn
        print(f"  {env_var}")
        print(f"  {arn}\n")
    except iam.exceptions.NoSuchEntityException:
        print(f"  {role_name} not found — deploy.py will create it (requires iam:CreateRole)\n")
    except Exception as e:
        print(f"  {role_name}: {e}\n")

---

## Step 2: Deploy

Runs all four runtimes (~3-5 min). The cell below executes `deploy.py` directly so it picks up the role ARNs set in the previous cell.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "deploy.py", "--name-prefix", "m4"],
    capture_output=False,
)
if result.returncode != 0:
    raise SystemExit(f"deploy.py failed with exit code {result.returncode}")

---

## Step 3: Review `main.py`

This is the code that runs inside the Runtime container.

In [ ]:
print(open("main.py").read())

---

## Step 4: Find the Runtime ARN

Run this cell to list all deployed Runtimes and find the ARN for this module.

In [ ]:
import os
import boto3

client_control = boto3.client("bedrock-agentcore-control", region_name=os.environ.get("AWS_REGION", "us-east-1"))
response = client_control.list_agent_runtimes()

print(f"{'Name':<40} {'ARN'}")
print("-" * 100)
for rt in response.get("agentRuntimes", []):
    print(f"{rt['agentRuntimeName']:<40} {rt['agentRuntimeArn']}")

---

## Step 5: Invoke the deployed agent

Paste the ARN from Step 4 into `RUNTIME_ARN` below, then run the cell.

In [ ]:
import os
import boto3, json, uuid
from botocore.config import Config

RUNTIME_ARN = ""  # paste ARN from Step 4 here

if not RUNTIME_ARN:
    raise ValueError("Set RUNTIME_ARN above before running this cell.")

client = boto3.client(
    "bedrock-agentcore",
    region_name=os.environ.get("AWS_REGION", "us-east-1"),
    config=Config(read_timeout=300),   # pipelines can take 60-180s
)

response = client.invoke_agent_runtime(
    agentRuntimeArn=RUNTIME_ARN,
    runtimeSessionId=str(uuid.uuid4()),
    payload=json.dumps({
        "prompt": "NovaCart Premium Tier: Options A ($19.99/mo invite-only), B ($14.99/mo 5% pilot), C ($12.99/mo full launch). Target: +15% CLV in 6 months."
    }).encode(),
    qualifier="DEFAULT",
)

result = json.loads(response["response"].read())
print(result.get("response", result))

---

## Step 6: Observability

After invoking, traces appear in **CloudWatch > X-Ray > Traces** or **Amazon Bedrock > AgentCore > Observability**.

What you see per invocation:
- Root span per `invoke_agent_runtime` call
- Child span per `Agent()` call inside the pipeline
- Tool call spans nested under each agent
- Duration breakdown per stage

No extra configuration needed: `aws-opentelemetry-distro` is in `requirements.txt` and AgentCore installs it automatically.

---

## Step 7: Cleanup

Delete all AWS resources created by this module.

```bash
python cleanup.py --name-prefix m4
```

Verify cleanup:

```bash
aws bedrock-agentcore-control list-agent-runtimes --region us-east-1
```